# Stage 7 -- 共表达基因模块 (Co-expression Gene Modules via hdWGCNA)

本 notebook 做共表达基因模块分析，识别在不同细胞状态中协同表达的
基因集合（模块）。主要依赖 hdWGCNA R 包（subprocess Rscript 桥接）。

## 生物学背景

**为什么做共表达模块？** 基因不是孤立工作的。在生物系统中，功能
相关的基因往往表现出协调的表达模式——同一通路的上游调控因子和
下游效应分子、同一蛋白复合物的各个亚基、同一代谢途径的连续酶，
在单细胞数据中呈现高度相关。

对胃"炎-癌"转化场景：
- **正常上皮稳态模块**：MUC5AC-MUC6-TFF1-TFF2 等黏液-保护因子共表达
- **肠化生模块**：CDX2-VIL1-MUC2-ALPI 等肠上皮标记协同激活
- **增殖/干性模块**：MKI67-PCNA-TOP2A + LGR5-SOX9-OLFM4 等分裂与干性基因
- **炎症-趋化模块**：CXCL-CXCR 家族 + NFKB 通路基因

**为什么用 hdWGCNA？** WGCNA（Weighted Gene Co-expression Network
Analysis）是共表达网络分析的金标准方法（Zhang & Horvath, 2005），
在 bulk 转录组领域被引用超过 20,000 次。hdWGCNA（Morabito et al.,
2023, Cell Genomics）是其在单细胞领域的扩展：
1. **Metacell 聚合**：先将同组内相似细胞聚合成"元细胞"（metacell），
   解决单细胞数据的稀疏性问题——单细胞 dropout 使直接相关不可靠
2. **软阈值加权网络**：通过幂函数将相关性转为连接强度（无标度拓扑）
3. **层次聚类模块检测**：Dynamic Tree Cut 自动识别模块
4. **模块特征基因 (ME)**：每个模块的第一主成分，总结该模块的整体表达

## hdWGCNA 三步流程

### 第一步：Metacell 构建
对每个分组内（如 Leiden cluster），将转录组相似的细胞聚合为 metacell。
每 20-30 个细胞 → 1 个 metacell，表达量为组内平均。目的：消除 dropout，
降低噪声，让共表达信号浮出水面。

### 第二步：共表达网络 + 模块识别
在 metacell 表达矩阵上计算基因间相关性 → 无标度拓扑转换（软阈值 β）→
TOM（拓扑重叠矩阵）→ 层次聚类 → Dynamic Tree Cut 切分模块。
模块即"在不同细胞状态下总是共波动"的基因集合。

### 第三步：模块特征基因 (ME) + 模块-性状关联
每个模块用其第一主成分（ME）概括——这是该模块"是否活跃"的单值代理。
ME 可与细胞性状（cluster、condition、pseudotime）做关联分析，
回答"哪些模块在哪些细胞群中特异活跃"。

## 本 notebook 的执行策略

- **Python 侧**：计算 HVG 基因对的 Spearman 相关矩阵并层次聚类——
  纯 Python（numpy/scipy），不依赖 R，可作为"朴素共表达模式"的预浏览
- **R/hdWGCNA 侧**：subprocess Rscript 桥接（ADR-0007）——完整的
  metacell→网络→模块检测→ME 流水线。若 hdWGCNA/WGCNA/Seurat R 包
  未安装，优雅跳过并给出清晰安装提示

**绝不编造模块**：hdWGCNA 不可用时跳过 R 部分，不产出假共表达模块。

产出：
- 基因相关性矩阵 → `results/tables/stage7_gene_correlation.csv`（Python）
- 模块分配表 → `results/tables/stage7_hdWGCNA_modules.csv`（R，若可用）
- 模块特征基因 → `results/tables/stage7_hdWGCNA_MEs.csv`（R，若可用）
- 可视化 → `results/figures/stage7_gene_modules_*`

> 参考：hdWGCNA 官方教程 https://smorabit.github.io/hdWGCNA/articles/basic_tutorial.html
> 无 student-code 参考——本模块为独立新增，按 ADR-0008 从零构建。

In [ ]:
# === PARAMS ===
# UPSTREAM_PATH              -- stage6 注释结果 h5ad
# OUTPUT_PATH                -- 本 notebook 产出 checkpoint
# LEIDEN_COL                 -- 分组 obs 列（metacell 按此列聚合；
#                               cell_type_final_v1 全 NaN 不可用）
# N_HVG                      -- 取 top HVG 基因数（控制共表达计算规模）
#                               WGCNA 推荐 >=2000，快速验证可用 500-1000
# TOP_K                      -- 每个基因保留 top-K 条最强相关基因对
#                               控制输出的链接表大小（全 2000x2000 上三角
#                               约 200 万对，全量 CSV 太大，只保留最强信号）
# HDWGCNA_WORK_DIR           -- hdWGCNA Rscript 临时工作目录
# HDWGCNA_MIN_MODULE_SIZE    -- hdWGCNA 最小模块基因数
# HDWGCNA_MERGE_CUT_HEIGHT   -- 模块合并高度阈值（越大合并越少）
# RSCRIPT_BIN                -- Rscript 可执行文件路径

UPSTREAM_PATH = "results/nancang_stage6_annotated_v1.h5ad"
OUTPUT_PATH   = "results/stage7_gene_modules.h5ad"

LEIDEN_COL = "leiden_res_0.6"

# 规模控制：HVG 子集基因数。WGCNA 标准建议 >=2000——当前 baseline
# HVG=2000 刚好满足；快速调试可降至 500。
N_HVG = 2000

# top-K 基因对比例：每个基因保留 TOP_K 条最强相关链接。
# 控制 stage7_gene_correlation_topK.csv 大小。
TOP_K = 20

HDWGCNA_WORK_DIR = "results/_hdWGCNA_tmp"
HDWGCNA_MIN_MODULE_SIZE  = 30
HDWGCNA_MERGE_CUT_HEIGHT = 0.25

RSCRIPT_BIN = "Rscript"

In [ ]:
# 确保框架 src/ 在 sys.path 并切换到项目根目录。
# 多级回退策略：nbconvert/conda run 的 CWD 不稳定，
# 先试 CWD，再试从 notebooks/stage7/ 回退两级，最后用 notebook 自身路径推算。
import sys, os, gc
_root = os.getcwd()
_root_candidates = [
    _root,
    os.path.abspath(os.path.join(_root, "..")),
    os.path.abspath(os.path.join(_root, "..", "..")),
]

for _cand in _root_candidates:
    if os.path.isdir(os.path.join(_cand, "src", "scrna_integration")):
        _root = _cand
        break
else:
    _root = os.environ.get("PROJECT_ROOT", _root)

if os.path.join(_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "src"))
os.chdir(_root)
os.makedirs("results/figures", exist_ok=True)
os.makedirs("results/tables", exist_ok=True)
os.makedirs(HDWGCNA_WORK_DIR, exist_ok=True)
print(f"PROJECT_ROOT: {_root}")
print(f"src 存在: {os.path.isdir(os.path.join(_root, 'src', 'scrna_integration'))}")

In [ ]:
# 导入依赖。
import scanpy as sc
import scipy.sparse as sp
import scipy.cluster.hierarchy as sch
import scipy.stats as stats
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import subprocess
import shutil
import warnings

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="anndata")
np.random.seed(42)

print(f"scanpy {sc.__version__}  |  numpy {np.__version__}  |  scipy {sp.__version__ if hasattr(sp, '__version__') else 'N/A'}")

In [ ]:
# 加载上游 stage6 输出。
# 契约：需包含 LEIDEN_COL 列 + highly_variable 标记。
# cell_type_final_v1 在本数据集全 NaN（1091/1091），回退到 LEIDEN_COL。
print(f"加载上游: {UPSTREAM_PATH}")
adata = sc.read_h5ad(UPSTREAM_PATH)
print(f"已加载: {adata.n_obs:,} 细胞 x {adata.n_vars:,} 基因")
print(f"X dtype: {adata.X.dtype}  |  sparse: {sp.issparse(adata.X)}")

# 检查分组列
if "cell_type_final_v1" in adata.obs.columns:
    _n_nan = adata.obs["cell_type_final_v1"].isna().sum()
    if _n_nan == adata.n_obs:
        # 全部为 NaN——统一注释尚未完成，回退到 Leiden 簇
        _group_col = LEIDEN_COL
        print(f"cell_type_final_v1 全 NaN ({_n_nan}/{adata.n_obs})，回退到 {LEIDEN_COL}")
    else:
        _group_col = "cell_type_final_v1"
        print(f"使用细胞类型列: cell_type_final_v1 (非 NaN: {adata.n_obs - _n_nan})")
elif LEIDEN_COL in adata.obs.columns:
    _group_col = LEIDEN_COL
    print(f"cell_type_final_v1 不存在，使用 {LEIDEN_COL}")
else:
    raise KeyError(f"缺少分组列: cell_type_final_v1 和 {LEIDEN_COL} 都不存在")

_groups = adata.obs[_group_col].astype(str)
_cluster_ids = sorted(_groups.unique())
print(f"分组列: {_group_col}  |  组数: {len(_cluster_ids)}")
print(f"各组成员数: {dict(zip(_cluster_ids, [_groups.eq(c).sum() for c in _cluster_ids]))}")

# 检查 HVG
if "highly_variable" in adata.var.columns:
    _n_hvg = adata.var["highly_variable"].sum()
    print(f"HVG 数: {_n_hvg}")
else:
    _n_hvg = 0
    print("WARNING: adata.var 无 'highly_variable' 列——将在全部基因上做共表达分析，"
          f"{adata.n_vars:,} 基因的 pairwise 相关矩阵可能很大。")

# 检查 layers / obsm
print(f"layers: {list(adata.layers.keys())}")
print(f"obsm keys: {list(adata.obsm.keys())}")

## hdWGCNA R 环境守卫

在运行 hdWGCNA 之前，无条件检查 R 环境和必需 R 包是否可用：
1. **Rscript 可执行文件** 是否存在
2. **hdWGCNA R 包** 是否已安装
3. **WGCNA R 包** 是否已安装（hdWGCNA 的依赖）
4. **Seurat R 包** 是否已安装（hdWGCNA 需要 Seurat 对象）

**为什么无条件检查？** 参考 grn.ipynb 的守卫模式——即使当前只跑 Python
侧的预处理，也把 R 环境状态检查清楚，避免 PI 忙活半天后发现跑不了
hdWGCNA。一次检查给出完整的"缺什么、怎么装"指引。

若 R 包不可用，以下 hdWGCNA 部分将优雅跳过。Python 侧的基因相关性
计算和聚类热图**不受影响**——这部分纯 Python，始终可运行。

In [ ]:
# === hdWGCNA R 环境守卫 ===
# 模式参考 grn.ipynb 的 pySCENIC cisTarget 守卫——
# 无条件检查所有前置条件，给出一次性完整的修复指引。

_r_available = shutil.which(RSCRIPT_BIN) is not None
print(f"Rscript 可用: {_r_available}  (RSCRIPT_BIN={RSCRIPT_BIN})")

_hdwgcna_available = False
_wgcna_available = False
_seurat_available = False

if _r_available:
    # 检查 hdWGCNA
    _check_cmds = {
        "hdWGCNA": "suppressPackageStartupMessages(library(hdWGCNA)); cat('OK')",
        "WGCNA":   "suppressPackageStartupMessages(library(WGCNA)); cat('OK')",
        "Seurat":  "suppressPackageStartupMessages(library(Seurat)); cat('OK')",
        "Matrix":  "suppressPackageStartupMessages(library(Matrix)); cat('OK')",
        "data.table": "suppressPackageStartupMessages(library(data.table)); cat('OK')",
    }

    _pkg_status = {}
    for _pkg, _r_code in _check_cmds.items():
        try:
            _res = subprocess.run(
                [RSCRIPT_BIN, "--vanilla", "-e", _r_code],
                capture_output=True, text=True, timeout=30,
            )
            _ok = _res.returncode == 0 and "OK" in _res.stdout
            _pkg_status[_pkg] = (_ok, _res.stderr.strip() if not _ok else "")
            print(f"  {_pkg}: {'OK' if _ok else 'MISSING'}"
                  f"{' -- ' + _res.stderr.strip()[:100] if not _ok else ''}")
        except (FileNotFoundError, subprocess.TimeoutExpired) as _e:
            _pkg_status[_pkg] = (False, str(_e))
            print(f"  {_pkg}: ERROR -- {_e}")

    _hdwgcna_available = _pkg_status.get("hdWGCNA", (False,))[0]
    _wgcna_available = _pkg_status.get("WGCNA", (False,))[0]
    _seurat_available = _pkg_status.get("Seurat", (False,))[0]
else:
    print("  Rscript 不可用——无法检查 R 包状态")

# 综合判断：三个核心包都可用才算就绪
_hdwgcna_ready = _r_available and _hdwgcna_available and _wgcna_available and _seurat_available

if not _hdwgcna_ready:
    _skip_lines = [
        "=" * 60,
        "hdWGCNA R 环境不完整——以下 hdWGCNA 子流程将优雅跳过。",
        "Python 侧的基因相关性计算和聚类热图不受影响，始终可运行。",
        "",
        "要启用 hdWGCNA 共表达模块分析，请按顺序安装以下 R 包:",
        "",
        "  # 1. 确认 R >= 4.1 已安装（conda: conda install -c conda-forge r-base）",
        "  # 2. 安装 BiocManager（用于安装 Bioconductor 包）",
        "  R -e 'install.packages(\"BiocManager\")'",
        "",
        "  # 3. 安装 Seurat 及其依赖",
        "  R -e 'install.packages(\"Seurat\")'",
        "",
        "  # 4. 安装 WGCNA",
        "  R -e 'BiocManager::install(\"WGCNA\")'",
        "",
        "  # 5. 安装 hdWGCNA（从 GitHub）",
        "  R -e 'install.packages(\"devtools\"); devtools::install_github(\"smorabit/hdWGCNA\")'",
        "",
        f"当前状态: Rscript={'Y' if _r_available else 'N'}, "
        f"hdWGCNA={'Y' if _hdwgcna_available else 'N'}, "
        f"WGCNA={'Y' if _wgcna_available else 'N'}, "
        f"Seurat={'Y' if _seurat_available else 'N'}",
        "",
        "安装完成后重跑本 notebook 即可启用完整 hdWGCNA 流程。",
        "=" * 60,
    ]
    print("\n".join(_skip_lines))
else:
    print("所有 R 包就绪——将完整运行 hdWGCNA 共表达模块分析。")

## 基因相关性矩阵（Python 侧，始终可运行）

**为什么先算相关性？** 在 hdWGCNA 做正式的网络分析之前，用纯 Python
（numpy/scipy）计算 HVG 基因对的 Spearman 相关系数矩阵。这有双重价值：
1. **预浏览**：即使 hdWGCNA 不可用，也能从相关性热图 + 层次聚类中
   看到基因的共表达模式——哪些基因族总是同涨同跌
2. **基线参照**：当 hdWGCNA 可用时，可对比"朴素 Spearman 相关"
   和"无标度网络+TOM"的模块划分差异——这是教学价值的体现

**为什么用 HVG 子集？** 全基因集（38606 基因）的 38606 x 38606 相关
矩阵约 5.9 GB 内存，且绝大多数低变异基因的相关性没有信息量。取 top
HVG 子集（2000 基因）的 2000 x 2000 矩阵约 32 MB，可在笔记本内直接
算完。

**为什么用 Spearman 而非 Pearson？** 单细胞数据的表达分布高度偏态
（大量零值 + 少数极高表达）。Spearman 秩相关对分布形态不敏感，
比 Pearson 更鲁棒地捕捉单调共表达关系——这在稀疏单细胞数据中更重要。

**为什么用秩变换+矩阵乘法？** naive 的 pairwise `spearmanr` 循环
对 2000 基因需要 O(n_genes^2 * n_cells) ≈ 4M * 1091 次操作，
耗时 ~30 分钟。秩变换技巧：对每个基因列做 rank 变换后，Pearson
相关 = Spearman rho。再通过归一化矩阵乘法 `X.T @ X` 一次性算出
全 2000x2000 相关矩阵——numpy/BLAS 在秒级完成。

In [ ]:
# === 基因相关性预计算（纯 Python，始终可运行） ===
# 取 top HVG 子集，在 log-normalized 表达矩阵上计算 pairwise
# Spearman 秩相关系数，然后层次聚类以发现共表达基因族。

# Step 1: 选取 HVG 子集
_n_hvg_use = min(N_HVG, _n_hvg if _n_hvg > 0 else adata.n_vars)
if "highly_variable" in adata.var.columns:
    _hvg_mask = adata.var["highly_variable"].values
    _hvg_idx = np.where(_hvg_mask)[0][:_n_hvg_use]
    _hvg_genes = adata.var_names[_hvg_idx].tolist()
else:
    _hvg_idx = np.arange(_n_hvg_use)
    _hvg_genes = adata.var_names[:_n_hvg_use].tolist()

print(f"共表达分析使用 {len(_hvg_genes)} 个 HVG 基因"
      f"（共 {adata.n_vars} 基因）")

# Step 2: 提取 log-normalized 表达矩阵（HVG 子集）
# 对相关性而言，log-normalized 比原始计数更合适——log 变换压缩了
# 少数极高表达的动态范围，让中等表达基因的相关也能被捕捉。
_X_hvg = adata[:, _hvg_genes].X
if sp.issparse(_X_hvg):
    _X_hvg_dense = _X_hvg.toarray()
else:
    _X_hvg_dense = np.asarray(_X_hvg)
print(f"HVG 表达矩阵: {_X_hvg_dense.shape[1]} 基因 x {_X_hvg_dense.shape[0]} 细胞"
      f"  (内存 {_X_hvg_dense.nbytes / 1e6:.1f} MB)")

# Step 3: 秩变换 + 矩阵乘法计算全 Spearman 相关矩阵
# 为什么这样算？对每个基因列做 rank 变换后，Pearson 相关 = Spearman rho。
# 然后通过归一化后的矩阵乘法 X.T @ X 一次性算出全相关矩阵——
# 利用 numpy/BLAS 的向量化，耗时从 ~30 分钟压缩到秒级。
# 秩变换操作约 2000 * 1091 * log(1091) ≈ 2M 次比较，秒级完成；
# 矩阵乘法 2000^2 * 1091 ≈ 4.4G FLOPs，BLAS 优化后在 2-5 秒。
print(f"秩变换 {_X_hvg_dense.shape[1]} 基因 x {_X_hvg_dense.shape[0]} 细胞...")
from scipy.stats import rankdata
_X_ranked = np.apply_along_axis(rankdata, 0, _X_hvg_dense)
# 中心化 + 归一化：使每一列（基因）均值为 0，L2 范数为 1
_X_centered = _X_ranked - _X_ranked.mean(axis=0)
_X_norms = np.sqrt((_X_centered ** 2).sum(axis=0))
_X_norms[_X_norms == 0] = 1.0  # 常量基因防除零
_X_normalized = _X_centered / _X_norms
# 矩阵乘法：R = X_normalized.T @ X_normalized（2000 x 2000）
# 这是全 Spearman rho 矩阵，每个元素 rho[i,j] = 基因 i 和基因 j 的秩相关
print(f"计算相关矩阵 ({_X_normalized.shape[1]} x {_X_normalized.shape[1]})...")
_rho_mat_full = _X_normalized.T @ _X_normalized  # (n_genes, n_genes)
# 由于浮点误差，对角线可能不是精确 1.0，修正之
np.fill_diagonal(_rho_mat_full, 1.0)
print(f"全 Spearman 相关矩阵: {_rho_mat_full.shape}  "
      f"(内存 {_rho_mat_full.nbytes / 1e6:.1f} MB)")

# Step 4: 从全矩阵提取 Top-K 基因对链接表
# 取上三角（避免重复），按 |rho| 排序，保留 top pairs。
# TOP_K（在 PARAMS cell 中定义）控制每个基因平均保留的对数。
_n_genes = _rho_mat_full.shape[0]
_triu_i, _triu_j = np.triu_indices(_n_genes, k=1)
_triu_rhos = _rho_mat_full[_triu_i, _triu_j]

# 取 |rho| 最大的 pairs（O(n^2 log n) 排序，~2M 元素，秒级）
_top_n_pairs = min(len(_triu_rhos), _n_genes * TOP_K)
_top_abs_idx = np.argpartition(np.abs(_triu_rhos), -_top_n_pairs)[-_top_n_pairs:]
_top_abs_idx = _top_abs_idx[np.argsort(np.abs(_triu_rhos[_top_abs_idx]))[::-1]]

_links = []
for _k in _top_abs_idx:
    _i, _j = _triu_i[_k], _triu_j[_k]
    _links.append((_hvg_genes[_i], _hvg_genes[_j], float(_triu_rhos[_k])))

_links_df = pd.DataFrame(_links, columns=["gene_a", "gene_b", "spearman_rho"])
_links_df = _links_df.sort_values("spearman_rho", key=abs, ascending=False)
print(f"Spearman 相关完成: {len(_links_df):,} 条基因对链接")
print(f"  rho 范围: [{_links_df['spearman_rho'].min():.4f}, "
      f"{_links_df['spearman_rho'].max():.4f}]")

# 保存链接表
_corr_csv = "results/tables/stage7_gene_correlation_topK.csv"
_links_df.to_csv(_corr_csv, index=False)
print(f"基因相关链接已保存: {_corr_csv} ({len(_links_df):,} 行)")

# Step 5: 对 top 200 基因做层次聚类（可视化用）
# 取 absolute rho 均值最高的 top 200 基因——这些是"最活跃的共表达节点"。
_gene_abs_mean = pd.Series(
    np.abs(_rho_mat_full).mean(axis=0),
    index=_hvg_genes,
).sort_values(ascending=False)

_top_n_clust = min(200, len(_gene_abs_mean))
_top_clust_genes = _gene_abs_mean.head(_top_n_clust).index.tolist()
_top_clust_idx = [list(_hvg_genes).index(g) for g in _top_clust_genes]

print(f"\n层次聚类使用 top {len(_top_clust_genes)} 共表达活跃基因")

# 从全矩阵取出子矩阵
_rho_mat = _rho_mat_full[np.ix_(_top_clust_idx, _top_clust_idx)].copy()
_rho_mat = _rho_mat.astype(np.float32)
del _rho_mat_full  # 释放在全矩阵上的引用（Python 侧变量，非内存泄漏）
_n_clust = len(_top_clust_genes)

# 层次聚类（Ward 法——最小化聚类内方差，优先产生紧凑模块）
_linkage = sch.linkage(
    1 - np.abs(_rho_mat),  # 距离 = 1 - |rho|
    method="ward",
)
# 切分成 4-8 个粗粒度的"朴素模块"（仅可视化用，无生物学声称）
_n_clusters = min(8, max(4, _n_clust // 30))
_cluster_labels = sch.fcluster(_linkage, _n_clusters, criterion="maxclust")
print(f"层次聚类完成: 切分为 {_n_clusters} 个粗粒度基因族")

In [ ]:
# === 基因相关性可视化（纯 Python，始终可运行） ===

# Fig 1: Spearman 相关分布直方图
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(_links_df["spearman_rho"], bins=80, color="#2ca02c",
             alpha=0.7, edgecolor="white", linewidth=0.3)
_q95 = _links_df["spearman_rho"].abs().quantile(0.95)
axes[0].axvline(_q95, color="red", ls="--", lw=1.2,
                label=f"|rho| 95% 分位 ({_q95:.3f})")
axes[0].axvline(-_q95, color="red", ls="--", lw=1.2)
axes[0].set_xlabel("Spearman rho")
axes[0].set_ylabel("Gene-pair Count")
axes[0].set_title("HVG Pairwise Spearman Correlation Distribution")
axes[0].legend(loc="upper right", frameon=False)

# Fig 2: Top 30 最强正相关 + Top 30 最强负相关基因对（条形图，精简）
_pos = _links_df.head(30)[::-1]
_neg = _links_df.tail(30)
axes[1].barh(range(len(_pos)), _pos["spearman_rho"].values,
             color="#d62728", alpha=0.7, label=f"Top {len(_pos)} 正相关")
axes[1].barh(range(len(_pos), len(_pos) + len(_neg)),
             _neg["spearman_rho"].values,
             color="#1f77b4", alpha=0.7, label=f"Top {len(_neg)} 负相关")
axes[1].set_yticks([])
axes[1].set_xlabel("Spearman rho")
axes[1].set_title("Top Positive & Negative Co-expression Gene Pairs")
axes[1].legend(loc="lower right", frameon=False)
axes[1].axvline(x=0, color="gray", lw=0.8)

plt.tight_layout()
_fig1 = "results/figures/stage7_gene_correlation_distribution.png"
fig.savefig(_fig1, dpi=200, bbox_inches="tight")
plt.close("all")
print(f"相关分布图已保存: {_fig1}")

# Fig 3: 层次聚类热图（top 200 基因的相关矩阵）
if len(_top_clust_genes) > 1:
    fig, ax = plt.subplots(figsize=(12, 11))
    # 按层次聚类顺序重排基因
    _order = sch.leaves_list(_linkage)
    _rho_ordered = _rho_mat[np.ix_(_order, _order)]

    sns.heatmap(
        _rho_ordered,
        cmap="RdBu_r", center=0,
        vmin=-1, vmax=1,
        xticklabels=False, yticklabels=False,
        cbar_kws={"label": "Spearman rho", "shrink": 0.6},
        ax=ax,
        rasterized=True,
    )
    ax.set_title(
        f"Gene Co-expression Clustering (top {len(_top_clust_genes)} HVGs)\n"
        f"Spearman rho | Ward linkage | {_n_clusters} rough clusters"
    )
    plt.tight_layout()
    _fig2 = "results/figures/stage7_gene_correlation_heatmap.png"
    fig.savefig(_fig2, dpi=200, bbox_inches="tight")
    plt.close("all")
    print(f"聚类热图已保存: {_fig2}")

# Fig 4: Top 基因对的共表达散点图（小 multiples——前 4 对）
_top4_pos = _links_df.head(4)
if len(_top4_pos) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(10, 9))
    axes = axes.flatten()
    for _i, (_, _row) in enumerate(_top4_pos.iterrows()):
        _ga, _gb, _r = _row["gene_a"], _row["gene_b"], _row["spearman_rho"]
        _ia = _hvg_genes.index(_ga) if _ga in _hvg_genes else 0
        _ib = _hvg_genes.index(_gb) if _gb in _hvg_genes else 1
        _ax = axes[_i]
        _ax.scatter(
            _X_hvg_dense[:, _ia], _X_hvg_dense[:, _ib],
            s=3, alpha=0.5, color="#2ca02c", rasterized=True,
        )
        _ax.set_xlabel(_ga, fontsize=8)
        _ax.set_ylabel(_gb, fontsize=8)
        _ax.set_title(f"Spearman rho = {_r:.3f}", fontsize=10)
    for _i in range(len(_top4_pos), 4):
        axes[_i].set_visible(False)
    plt.tight_layout()
    _fig3 = "results/figures/stage7_gene_coexpression_scatter.png"
    fig.savefig(_fig3, dpi=200, bbox_inches="tight")
    plt.close("all")
    print(f"共表达散点图已保存: {_fig3}")

## hdWGCNA -- Metacell 构建 + 共表达网络 + 模块检测

**为什么用 subprocess Rscript？** hdWGCNA 的完整流水线依赖 Seurat
对象、WGCNA 的无标度网络、Dynamic Tree Cut 等 R 生态组件。通过
rpy2 桥接 Seurat 的大对象在依赖升级时极易崩溃（ADR-0007）。
subprocess Rscript 模式更鲁棒：写 .mtx/.csv → `Rscript --vanilla` →
读结果 CSV + PNG。R 脚本独立可调试，不污染 Python 环境。

**hdWGCNA 流水线概览**（R 侧执行）：
1. 创建 Seurat 对象（从 .mtx 计数矩阵）
2. `SetupForWGCNA`——选择 variable genes 作为 WGCNA 输入
3. `MetacellsByGroups`——按分组列（Leiden cluster）聚合相似细胞
4. `TestSoftPowers`——确定最优软阈值 β（满足无标度拓扑 R^2 > 0.8）
5. `ConstructNetwork`——构建加权共表达网络 + 模块检测
6. `ModuleEigengenes`——计算每个模块的 ME
7. 输出：模块分配表、ME 矩阵、dendrogram 图、软阈值诊断图

**数据导出格式**：
- 计数矩阵 `.mtx`（genes x cells，Monocle3/UCell 同款格式）
- 细胞元数据 `.csv`（含 cell_id + 分组列）
- 基因注释 `.csv`（gene_id + gene_short_name）

In [ ]:
# === hdWGCNA 数据导出 ===
# 写 .mtx + .csv 供 R 脚本读取。格式与 pseudotime.ipynb 的 Monocle3
# 导出完全一致——genes x cells 的 counts 矩阵 + cell_meta + gene_anno。

if not _hdwgcna_ready:
    print("hdWGCNA R 环境不可用——跳过数据导出。")
    _hdwgcna_exported = False
else:
    shutil.rmtree(HDWGCNA_WORK_DIR, ignore_errors=True)
    os.makedirs(HDWGCNA_WORK_DIR, exist_ok=True)

    # 获取计数矩阵（优先 counts layer，其次 raw，最后 .X）
    # 注意：hdWGCNA 需要 counts（原始整数计数），
    # 但此数据集的 counts layer 存的是 float32。R 侧 Seurat 会做
    # NormalizeData——接受 float 作为输入没有问题。
    if "counts" in adata.layers:
        _X_export = adata.layers["counts"]
    elif adata.raw is not None:
        _X_export = adata.raw[:, adata.var_names].X
    else:
        _X_export = adata.X

    # 写 .mtx（genes x cells——R 侧 Seurat 需要基因在行）
    from scipy.io import mmwrite
    _X_gc = sp.csr_matrix(_X_export).T.tocoo()
    _mtx_path = os.path.join(HDWGCNA_WORK_DIR, "counts.mtx")
    mmwrite(_mtx_path, _X_gc)
    print(f"计数矩阵已导出: {_mtx_path}  (shape genes x cells: {_X_gc.shape})")

    # 细胞元数据（含 cell_id + 分组列）
    _meta = adata.obs[[_group_col]].copy()
    _meta["cell_id"] = adata.obs_names.astype(str)
    for _col in _meta.columns:
        if hasattr(_meta[_col], "cat"):
            _meta[_col] = _meta[_col].astype(str)
    _meta_path = os.path.join(HDWGCNA_WORK_DIR, "cell_meta.csv")
    _meta.to_csv(_meta_path, index=False)
    print(f"细胞元数据已导出: {_meta_path}  "
          f"({_meta.shape[0]} 细胞 x {_meta.shape[1]} 列)")

    # 基因注释
    _gene_df = pd.DataFrame({
        "gene_id": adata.var_names.astype(str),
        "gene_short_name": adata.var_names.astype(str),
    })
    _gene_path = os.path.join(HDWGCNA_WORK_DIR, "gene_anno.csv")
    _gene_df.to_csv(_gene_path, index=False)
    print(f"基因注释已导出: {_gene_path}  ({len(_gene_df)} 基因)")

    _hdwgcna_exported = True

In [ ]:
# === hdWGCNA R 脚本——内联生成然后 subprocess 调用 ===
# 参考 hdWGCNA 官方教程 (https://smorabit.github.io/hdWGCNA/articles/
# basic_tutorial.html)，完整流程：Seurat → SetupForWGCNA →
# MetacellsByGroups → TestSoftPowers → ConstructNetwork →
# ModuleEigengenes → ModuleConnectivity → 导出。
#
# 设计要点:
# - 软阈值自动选择：优先取 R^2 > 0.8 的最小 power，达不到则取最佳
# - 最小模块 30 基因：过滤掉噪音小的碎片模块
# - 每一步都写中间 CSV——方便排查问题
# - 输出 PNG 图供 notebook 直接展示

if not (_hdwgcna_ready and _hdwgcna_exported):
    print("hdWGCNA 条件不满足——跳过 R 脚本生成。")
    _hdwgcna_success = False
else:
    _r_script = os.path.join(HDWGCNA_WORK_DIR, "run_hdWGCNA.R")
    _r_code = '''#!/usr/bin/env Rscript
suppressPackageStartupMessages({
  library(Seurat)
  library(hdWGCNA)
  library(WGCNA)
  library(Matrix)
  library(data.table)
  library(ggplot2)
})

args <- commandArgs(trailingOnly = TRUE)
input_mtx     <- args[1]   # counts.mtx (genes x cells)
input_meta    <- args[2]   # cell_meta.csv
input_gene    <- args[3]   # gene_anno.csv
output_prefix <- args[4]   # output prefix
group_col     <- args[5]   # grouping column name
n_hvg         <- as.integer(args[6])  # number of HVGs
min_module    <- as.integer(args[7])  # min module size
merge_height  <- as.numeric(args[8])  # merge cut height

cat("=== hdWGCNA pipeline start ===", "\n")
cat("Reading data...", "\n")

# ---- 1. Read data ----
expr <- readMM(input_mtx)          # genes x cells
gene_anno <- fread(input_gene, data.table = FALSE)
cell_meta <- fread(input_meta, data.table = FALSE)
rownames(cell_meta) <- cell_meta$cell_id
rownames(gene_anno) <- gene_anno$gene_id

# 匹配细胞 ID
colnames(expr) <- cell_meta$cell_id
rownames(expr) <- gene_anno$gene_id
cat("  Matrix dim:", dim(expr), "\n")
cat("  Cells:", ncol(expr), "  Genes:", nrow(expr), "\n")

# ---- 2. Create Seurat object ----
cat("Creating Seurat object...", "\n")
seurat_obj <- CreateSeuratObject(counts = expr, meta.data = cell_meta)
seurat_obj <- NormalizeData(seurat_obj, verbose = FALSE)
seurat_obj <- FindVariableFeatures(seurat_obj, nfeatures = n_hvg, verbose = FALSE)
seurat_obj <- ScaleData(seurat_obj, verbose = FALSE)

# ---- 3. hdWGCNA Setup ----
cat("Setting up hdWGCNA...", "\n")
seurat_obj <- SetupForWGCNA(
  seurat_obj,
  gene_select = "variable",
  wgcna_name = "scrna_modules"
)

# ---- 4. Metacell construction ----
# metacell 是 hdWGCNA 的核心创新：将同组内转录组相似的细胞聚合为
# 伪细胞（pseudocell），消除单细胞 dropout 噪声。
# k=25 表示每个 metacell 聚合约 25 个相似细胞。
# max_shared=10 控制不同组之间共享细胞的上限。
cat("Constructing metacells by", group_col, "...", "\n")
seurat_obj <- MetacellsByGroups(
  seurat_obj,
  group.by = c(group_col),
  reduction = "pca",
  k = 25,
  max_shared = 10,
  ident.group = group_col
)
cat("  Metacells:", ncol(seurat_obj@misc$scrna_modules$wgcna_metacell_obj), "\n")

# ---- 5. Set expression to metacells ----
# 在 metacell 表达矩阵而非原始稀疏矩阵上运行 WGCNA
_groups <- unique(seurat_obj@meta.data[[group_col]])
seurat_obj <- SetDatExpr(
  seurat_obj,
  group_name = as.character(_groups),
  group.by = group_col
)

# ---- 6. Soft power threshold ----
cat("Testing soft power thresholds...", "\n")
seurat_obj <- TestSoftPowers(seurat_obj, networkType = "signed")

# 自动选择软阈值：优先满足无标度拓扑 R^2 > 0.8，若达不到则取最接近的
sp_table <- GetPowerTable(seurat_obj)
write.csv(sp_table, paste0(output_prefix, "_soft_power_table.csv"),
          row.names = FALSE)
cat("  Soft power table written.", "\n")

sp_idx <- which(sp_table$SFT.R.sq > 0.8)
if (length(sp_idx) > 0) {
  soft_power_val <- sp_table$Power[min(sp_idx)]
  cat("  Selected soft power:", soft_power_val,
      "(R^2 =", sp_table$SFT.R.sq[min(sp_idx)], ")", "\n")
} else {
  soft_power_val <- sp_table$Power[which.max(sp_table$SFT.R.sq)]
  cat("  WARNING: No power reaches R^2 > 0.8. Using best:", soft_power_val,
      "(R^2 =", max(sp_table$SFT.R.sq), ")", "\n")
}

# ---- 7. Construct co-expression network & detect modules ----
cat("Constructing network (soft_power =", soft_power_val,
    ", min_module =", min_module, ")...", "\n")
seurat_obj <- ConstructNetwork(
  seurat_obj,
  soft_power = soft_power_val,
  min_module_size = min_module,
  merge_cut_height = merge_height,
  tom_name = "scrna",
  networkType = "signed"
)

# ---- 8. Module eigengenes & connectivity ----
cat("Computing module eigengenes...", "\n")
seurat_obj <- ModuleEigengenes(seurat_obj)
seurat_obj <- ModuleConnectivity(seurat_obj)

# ---- 9. Export results ----
cat("Exporting results...", "\n")

# Module assignments (gene -> module mapping)
modules <- GetModules(seurat_obj)
write.csv(modules, paste0(output_prefix, "_modules.csv"), row.names = FALSE)
cat("  Modules written:", nrow(modules), "genes", "\n")

# Module eigengenes (MEs) -- metacell-level
mes <- GetMEs(seurat_obj)
write.csv(mes, paste0(output_prefix, "_MEs.csv"))
cat("  MEs written:", nrow(mes), "x", ncol(mes), "\n")

# Module summary
module_summary <- as.data.frame(table(modules$module))
colnames(module_summary) <- c("module", "n_genes")
write.csv(module_summary, paste0(output_prefix, "_module_summary.csv"),
          row.names = FALSE)
cat("  Module summary:", nrow(module_summary), "modules", "\n")

# Hub genes per module
hub_df <- GetHubGenes(seurat_obj, n_hubs = 10)
write.csv(hub_df, paste0(output_prefix, "_hub_genes.csv"), row.names = FALSE)
cat("  Hub genes written.", "\n")

# ---- 10. Figures ----
cat("Generating figures...", "\n")

png(paste0(output_prefix, "_dendrogram.png"),
    width = 2400, height = 1600, res = 200)
ModuleDendrogram(seurat_obj, main = "hdWGCNA Gene Co-expression Modules")
dev.off()
cat("  Dendrogram saved.", "\n")

png(paste0(output_prefix, "_soft_power_diagnostic.png"),
    width = 2000, height = 1200, res = 200)
print(PlotSoftPowers(seurat_obj))
dev.off()
cat("  Soft power plot saved.", "\n")

cat("=== hdWGCNA pipeline completed ===", "\n")
'''

    # 写 R 脚本
    with open(_r_script, "w", encoding="utf-8") as _f:
        _f.write(_r_code)
    print(f"R 脚本已生成: {_r_script}")

    # 构建调用命令
    _mtx_path = os.path.join(HDWGCNA_WORK_DIR, "counts.mtx")
    _meta_path_val = os.path.join(HDWGCNA_WORK_DIR, "cell_meta.csv")
    _gene_path_val = os.path.join(HDWGCNA_WORK_DIR, "gene_anno.csv")
    _out_prefix = os.path.join(HDWGCNA_WORK_DIR, "hdWGCNA")

    _cmd = [
        RSCRIPT_BIN, "--vanilla", _r_script,
        _mtx_path,
        _meta_path_val,
        _gene_path_val,
        _out_prefix,
        _group_col,
        str(N_HVG),
        str(HDWGCNA_MIN_MODULE_SIZE),
        str(HDWGCNA_MERGE_CUT_HEIGHT),
    ]
    print(f"正在运行: {' '.join(_cmd)}")

    _env = os.environ.copy()
    _env["R_PROFILE_USER"] = ""
    _env["R_ENVIRON_USER"] = ""

    try:
        _res = subprocess.run(
            _cmd, capture_output=True, text=True,
            env=_env, timeout=1800,
        )
        # 保存 stdout/stderr
        _stdout_path = os.path.join(HDWGCNA_WORK_DIR, "stdout.log")
        _stderr_path = os.path.join(HDWGCNA_WORK_DIR, "stderr.log")
        with open(_stdout_path, "w") as _f:
            _f.write(_res.stdout or "")
        with open(_stderr_path, "w") as _f:
            _f.write(_res.stderr or "")

        if _res.returncode != 0:
            print(f"hdWGCNA 运行失败 (exitcode={_res.returncode})")
            print(f"  STDOUT: {_stdout_path}")
            print(f"  STDERR: {_stderr_path}")
            if _res.stderr:
                print(f"  STDERR 尾部: {_res.stderr[-300:]}")
            _hdwgcna_success = False
        else:
            print("hdWGCNA 运行成功")
            # 显示关键摘要
            _r_out = _res.stdout
            for _line in _r_out.split("\n"):
                if any(_kw in _line for _kw in [
                    "Metacells", "Selected soft power",
                    "Modules written", "MEs written",
                    "Module summary", "completed",
                ]):
                    print(f"  [R] {_line.strip()}")
            _hdwgcna_success = True
    except subprocess.TimeoutExpired:
        print("hdWGCNA 运行超时（>30 分钟），已终止")
        _hdwgcna_success = False

    del _res, _env

In [ ]:
# === 读取 hdWGCNA 结果，写回 adata.uns ===

_hdwgcna_done = False
if _hdwgcna_ready and _hdwgcna_exported and _hdwgcna_success:
    _modules_csv = os.path.join(HDWGCNA_WORK_DIR, "hdWGCNA_modules.csv")
    _mes_csv = os.path.join(HDWGCNA_WORK_DIR, "hdWGCNA_MEs.csv")
    _summary_csv = os.path.join(HDWGCNA_WORK_DIR, "hdWGCNA_module_summary.csv")
    _hub_csv = os.path.join(HDWGCNA_WORK_DIR, "hdWGCNA_hub_genes.csv")

    if os.path.exists(_modules_csv):
        _modules_df = pd.read_csv(_modules_csv)
        print(f"模块分配表: {_modules_df.shape[0]} 基因 -> {_modules_df['module'].nunique()} 模块")
        print(f"  模块分布: {dict(_modules_df['module'].value_counts().head(10))}")
        adata.uns["stage7_hdWGCNA_modules"] = _modules_df.to_dict(orient="list")
    else:
        print(f"模块分配表不存在: {_modules_csv}")

    if os.path.exists(_summary_csv):
        _summary_df = pd.read_csv(_summary_csv)
        print(f"模块摘要: {len(_summary_df)} 个模块")
        if len(_summary_df) > 0:
            print(f"  最大模块: {_summary_df['module'].iloc[0]} "
                  f"({_summary_df['n_genes'].iloc[0]} genes)")
        adata.uns["stage7_hdWGCNA_summary"] = _summary_df.to_dict(orient="list")

    if os.path.exists(_hub_csv):
        _hub_df = pd.read_csv(_hub_csv)
        print(f"Hub genes: {_hub_df.shape[0]} 行")
        adata.uns["stage7_hdWGCNA_hub_genes"] = _hub_df.to_dict(orient="list")

    # 复制 R 产出的 figure 到 results/figures/
    for _fname in [
        "hdWGCNA_dendrogram.png",
        "hdWGCNA_soft_power_diagnostic.png",
    ]:
        _src = os.path.join(HDWGCNA_WORK_DIR, _fname)
        if os.path.exists(_src):
            _dst = os.path.join("results", "figures", f"stage7_{_fname}")
            shutil.copy2(_src, _dst)
            print(f"hdWGCNA figure 已复制: {_dst}")

    _hdwgcna_done = True
else:
    print("hdWGCNA 结果不可用——"
          f"ready={_hdwgcna_ready}, exported={_hdwgcna_exported if '_hdwgcna_exported' in dir() else False}, "
          f"success={_hdwgcna_success if '_hdwgcna_success' in dir() else False}")
    print("adata.uns 不写入 hdWGCNA 结果。Python 侧的相关性结果已写入 adata.uns。")

## 结果汇总与 checkpoint

将本 notebook 的运行元数据写入 `adata.uns['stage7_gene_modules_v1']`。
无论 hdWGCNA 是否成功运行，Python 侧的相关性结果始终写入。

In [ ]:
# 运行摘要与元数据写入 adata.uns。
import datetime as _dt

_stage7_mod_uns = {
    "method": "hdWGCNA (subprocess Rscript) + Python Spearman correlation",
    "group_col": _group_col,
    "n_hvg_used": _n_hvg_use,
    "n_corr_links": len(_links_df),
    "hdwgcna_r_available": _r_available,
    "hdwgcna_pkg_available": _hdwgcna_available,
    "wgcna_pkg_available": _wgcna_available,
    "seurat_pkg_available": _seurat_available,
    "hdwgcna_ran": _hdwgcna_done if "_hdwgcna_done" in dir() else False,
    "hdwgcna_success": _hdwgcna_success if "_hdwgcna_success" in dir() else False,
    "timestamp": _dt.datetime.now().isoformat(),
}

_stage7_mod_uns["hdwgcna_params"] = {
    "min_module_size": str(HDWGCNA_MIN_MODULE_SIZE),
    "merge_cut_height": str(HDWGCNA_MERGE_CUT_HEIGHT),
    "work_dir": str(HDWGCNA_WORK_DIR),
}

# Python 侧相关性结果摘要（用简单字符串列表，避免 h5ad 序列化问题）
if len(_links_df) > 0:
    _top_pos = _links_df.head(10)
    _top_neg = _links_df.tail(10)
    _stage7_mod_uns["correlation_top_positive"] = [
        f"{r['gene_a']}|{r['gene_b']}|{float(r['spearman_rho']):.4f}"
        for _, r in _top_pos.iterrows()
    ]
    _stage7_mod_uns["correlation_top_negative"] = [
        f"{r['gene_a']}|{r['gene_b']}|{float(r['spearman_rho']):.4f}"
        for _, r in _top_neg.iterrows()
    ]

adata.uns["stage7_gene_modules_v1"] = _stage7_mod_uns
print("运行元数据已写入 adata.uns['stage7_gene_modules_v1']")
print()
print("=" * 50)
print("Stage 7 共表达基因模块 执行摘要:")
print(f"  Python Spearman 相关: 已完成 ({len(_links_df):,} gene pairs, "
      f"{_n_hvg_use} HVG genes)")
print(f"  hdWGCNA R 环境:  "
      f"Rscript={'Y' if _r_available else 'N'}, "
      f"hdWGCNA={'Y' if _hdwgcna_available else 'N'}, "
      f"WGCNA={'Y' if _wgcna_available else 'N'}, "
      f"Seurat={'Y' if _seurat_available else 'N'}")
print(f"  hdWGCNA 运行:    "
      f"{'已完成' if ('_hdwgcna_done' in dir() and _hdwgcna_done) else '跳过'}")
print(f"  Python 可视化:   {'已产出' if len(_links_df) > 0 else '无数据'}")
print("=" * 50)

In [ ]:
# 内存自检——确保 adata.X 稀疏性/精度未被破坏。
assert sp.issparse(adata.X) and adata.X.dtype == np.float32, (
    f"adata.X 不变量被破坏: sparse={sp.issparse(adata.X)}, dtype={adata.X.dtype}"
)
print("内存自检通过: X 是 sparse CSR float32")

# 检查新增 uns 数据
if "stage7_gene_modules_v1" in adata.uns:
    _keys = list(adata.uns["stage7_gene_modules_v1"].keys())
    print(f"  stage7_gene_modules_v1 keys: {_keys}")

if "stage7_hdWGCNA_modules" in adata.uns:
    _mods = adata.uns["stage7_hdWGCNA_modules"]
    print(f"  hdWGCNA modules: {len(_mods)} entries")
else:
    print("  hdWGCNA modules: 未写入（hdWGCNA 未运行或失败）")

In [ ]:
# 写出 checkpoint。
adata.write_h5ad(OUTPUT_PATH, compression="lzf")
print(f"已写出 {OUTPUT_PATH}")
assert os.path.exists(OUTPUT_PATH), f"输出不存在: {OUTPUT_PATH}"
print(f"已验证: {OUTPUT_PATH} ({os.path.getsize(OUTPUT_PATH):,} bytes)")

In [ ]:
# 释放内存。
del adata, _links_df, _X_hvg_dense
gc.collect()
print("内存已释放")